# RED & WHITE – Risk Alert Classifier Exam (Complete Solution)

## Objective
Build, evaluate and optimize a classification model to predict **risk_status**.

### Parts Covered
- Part A: Conceptual Understanding
- Part B: Dataset Understanding & Preparation
- Part C: Baseline Classification Model
- Part D: Handling Imbalanced Data
- Part E: Tree-Based Models
- Part F: Hyperparameter Tuning
- Part G: Model Evaluation & ROC Analysis
- Part H: Final Analysis & Reporting


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

from imblearn.over_sampling import SMOTE


# Part A – Conceptual Understanding

### Q1 What is Logistic Regression?
Used for binary classification by predicting probability.

### Q2 Precision vs Recall
Precision = Correct positive predictions / Total predicted positives

Recall = Correct positive predictions / Actual positives

### Q3 Explain SMOTE
Creates synthetic samples for minority class.

### Q4 Precision, Recall, F1, TPR, FPR
Important classification evaluation metrics.

### Q5 ROC Curve
Graph between TPR and FPR across thresholds.


# Part B – Dataset Understanding & Preparation

In [ ]:
df = pd.read_csv('Risk_Alert_Classifier_Dataset_4600 - Risk_Alert_Classifier_Dataset_4600.csv.csv')

print("Dataset Shape:", df.shape)
df.head()


In [ ]:
# Missing values
df.isnull().sum()


In [ ]:
# Dataset information
df.info()


In [ ]:
# Target class distribution
df['risk_status'].value_counts()


In [ ]:
# Encode categorical columns

data = df.copy()

for col in data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

X = data.drop('risk_status', axis=1)
y = data['risk_status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)


# Part C – Baseline Classification Model

### Logistic Regression
Tasks Covered:
- Train model
- Evaluate accuracy


In [ ]:
log_model = LogisticRegression(max_iter=5000)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, log_pred))


# Part D – Handling Imbalanced Data

Tasks Covered:
1. Check imbalance
2. Apply SMOTE
3. Retrain model
4. Compare Precision, Recall and F1


In [ ]:
print("Before SMOTE")
print(y_train.value_counts())

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE")
print(y_train_smote.value_counts())


In [ ]:
log_smote = LogisticRegression(max_iter=5000)

log_smote.fit(X_train_smote, y_train_smote)

pred_smote = log_smote.predict(X_test)

print("Precision:", precision_score(y_test, pred_smote))
print("Recall:", recall_score(y_test, pred_smote))
print("F1:", f1_score(y_test, pred_smote))


# Part E – Tree Based Classification Models

Tasks Covered:
- Decision Tree
- Random Forest
- Accuracy Comparison


In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train_smote, y_train_smote)

dt_pred = dt.predict(X_test)

dt_acc = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy:", dt_acc)


In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)


In [ ]:
comparison = pd.DataFrame({
    'Model':['Decision Tree','Random Forest'],
    'Accuracy':[dt_acc, rf_acc]
})

comparison


# Part F – Hyperparameter Tuning

Tasks Covered:
- Apply GridSearchCV
- Tune Random Forest
- Compare performance


In [ ]:
param_grid = {
    'n_estimators':[100,200],
    'max_depth':[5,10,None],
    'min_samples_split':[2,5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_smote, y_train_smote)

print("Best Parameters")
print(grid.best_params_)

best_rf = grid.best_estimator_


In [ ]:
best_pred = best_rf.predict(X_test)

print("Tuned Accuracy:", accuracy_score(y_test, best_pred))


# Part G – Model Evaluation & ROC Analysis

Tasks Covered:
- Confusion Matrix
- Classification Report
- ROC Curve
- ROC-AUC Score


In [ ]:
cm = confusion_matrix(y_test, best_pred)

print(cm)


In [ ]:
print(classification_report(y_test, best_pred))


In [ ]:
prob = best_rf.predict_proba(X_test)[:,1]

fpr, tpr, threshold = roc_curve(y_test, prob)

auc_score = roc_auc_score(y_test, prob)

plt.figure(figsize=(6,4))
plt.plot(fpr,tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

print("ROC-AUC:", auc_score)


# Part H – Final Analysis & Reporting

## Findings
1. Dataset loaded and prepared.
2. Logistic Regression built as baseline.
3. SMOTE handled imbalance.
4. Decision Tree and Random Forest trained.
5. GridSearchCV improved Random Forest.
6. ROC-AUC and Classification Report evaluated.

## Business Recommendation
Deploy the tuned Random Forest model for risk alert prediction because it provides stronger classification capability and better generalization.

## Submission Checklist
- Dataset analysis completed
- Missing values checked
- Logistic Regression completed
- SMOTE applied
- Decision Tree completed
- Random Forest completed
- Hyperparameter tuning completed
- Confusion Matrix completed
- ROC Curve completed
- Final recommendations completed
